[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/04_cell_specific_six_sweep_fitting.ipynb)

In [ ]:
# Colab / local repository setup
from pathlib import Path
import os, subprocess, sys


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


def _run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


REPO_URL = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
REPO_BRANCH = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
PROJECT_DIRNAME = os.environ.get("ASTROMODEL_PROJECT_DIRNAME", "astromodel_proving")

if _running_in_colab():
    project_root = Path('/content') / PROJECT_DIRNAME
    if not project_root.exists():
        try:
            _run(['git','clone','--depth','1','--branch',REPO_BRANCH,REPO_URL,str(project_root)])
        except subprocess.CalledProcessError:
            _run(['git','clone','--depth','1',REPO_URL,str(project_root)])
    os.chdir(project_root)
    requirements = project_root / 'requirements.txt'
    if requirements.exists():
        _run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    project_root = next((c for c in candidates if (c / 'src').is_dir() and (c / 'data').is_dir()), current)
    os.chdir(project_root)

os.environ['ASTROMODEL_PROJECT_ROOT'] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'ASTROMODEL_PROJECT_ROOT={project_root}')
print(f'Working directory={Path.cwd()}')
print(f'data exists={(project_root / "data").exists()}, src exists={(project_root / "src").exists()}')

# Step 04 — Cell-specific six-sweep fitting and accepted ensemble construction

This notebook validates Step 04 using the **expected reviewer-facing astrocyte ODE model**.

Scope of this notebook:
- verify that the implemented `src.astro_model.model` matches the expected equations;
- build cell-specific six-sweep fits under that model;
- use Step 02 region-aware thresholds to define accepted ensembles;
- run held-out-sweep screening as part of the reviewer-facing contract.

Claim boundary:
- this notebook creates accepted cell-specific ensembles;
- it does **not** by itself claim biological degeneracy;
- mechanistic decomposition belongs to Step 05;
- predictive robustness beyond held-out sweeps belongs to Step 06.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

from src.astro_model import build_paramdict, model
from src.step04_cell_fits import acceptance_contract_table

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
PROJECT_ROOT = Path(os.environ.get('ASTROMODEL_PROJECT_ROOT', Path.cwd())).resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'cell_fits_step04_model_aligned_demo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

## Model-alignment audit

In [ ]:
def reference_model(z, t, paramdict):
    Cm_a  = paramdict["Astrocyte"]["Cm_a"]
    g_kir = paramdict["Astrocyte"]["g_kir"]
    A = paramdict["Astrocyte"]["A"]
    g_k_a = paramdict["Astrocyte"]["g_k_a"]
    gl_a = paramdict["Astrocyte"]["gl_a"]
    w_a = paramdict["Astrocyte"]["w_a"]
    K_a0 = paramdict["Astrocyte"]["K_a0"]
    Sig_a = paramdict["Astrocyte"]["Sig_a"]
    gama_t = paramdict["Astrocyte"]["gama_t"]
    gama_s = paramdict["Astrocyte"]["gama_s"]
    Z_th = paramdict["Astrocyte"]["Z_th"]
    Z_s = paramdict["Astrocyte"]["Z_s"]
    Va_0 = paramdict["Astrocyte"]["Va_0"]
    Va_s = paramdict["Astrocyte"]["Va_s"]
    Va_l = paramdict["Astrocyte"]["Va_l"]
    P_k = paramdict["Astrocyte"]["P_k"]
    d_gap = paramdict["Astrocyte"]["d_gap"]
    F = paramdict["Astrocyte"]["F"]
    R = paramdict["Astrocyte"]["R"]
    T = paramdict["Astrocyte"]["T"]
    K_o0 =paramdict["external"]["K_o0"]
    w_o = paramdict["external"]["w_o"]
    epsilon = paramdict["external"]["epsilon"]
    idx = np.where(paramdict["external"]["K_bath"]["time"]<=t)[0][-1]
    K_bath = paramdict["external"]["K_bath"]["value"][idx]
    switching_function = paramdict["Astrocyte"].get("switching_function", "sigmoid")
    if "epsilon_middle" in paramdict["external"] and idx == 1:
      epsilon = epsilon*paramdict["external"]["epsilon_middle"]
    if "w_o_middle" in paramdict["external"] and idx == 1:
      w_o = w_o*paramdict["external"]["w_o_middle"]
    Va  = z[0]
    DK_a_t = z[1]
    K_a_s = z[2]
    Kg = z[3]
    DK_a = DK_a_t + K_a_s
    K_a  = K_a0 +DK_a
    DK_o_a = -(w_a/w_o)*DK_a_t
    K_o  = K_o0 + DK_o_a + Kg
    K_ratio = K_o / K_a
    if K_ratio <= 0: K_ratio = 1e-8
    E_k_a = 25.7 * np.log(K_ratio)
    I_k_a = g_k_a*(Va - E_k_a)
    I_Kir = g_kir * np.sqrt(np.abs(K_o))*(Va - E_k_a)*(1/(1+np.exp((Va - E_k_a)/19.2)))
    PH_a = 0.04*(Va - Va_s)
    P_kgap = d_gap*P_k
    exp_neg_PH_a = np.exp(-PH_a)
    denominator = -1 + np.exp(-PH_a)
    if denominator == 0: denominator = 1e-8
    I_kgap = P_kgap * F * PH_a * (1 / denominator) * ((K_a * exp_neg_PH_a) - K_a0)
    I_l_a  = gl_a*(Va - Va_l)
    if switching_function == "sigmoid":
        Th_s = DK_a / (1 + np.exp((Z_th - DK_a_t) * Z_s))
    elif switching_function == "tanh":
        Th_s = DK_a * (0.5 * (1 + np.tanh((DK_a_t - Z_th) * Z_s)))
    elif switching_function == "hill":
        n = paramdict["Astrocyte"].get("hill_coefficient", 2)
        K_d = paramdict["Astrocyte"].get("K_d", 1)
        Th_s = DK_a * ((DK_a_t ** n) / (K_d ** n + DK_a_t ** n))
    else:
        raise ValueError(f"Unknown switching function type: {switching_function}")
    dVa   = (-1.0/Cm_a)*(I_Kir + I_k_a +I_l_a +I_kgap)
    dDK_a_t = -(gama_t*Sig_a/(w_a*F))*(I_Kir + I_k_a)
    dK_a_s = -Th_s*(gama_s*Sig_a/(w_a*F))* I_kgap
    dKg   =  epsilon*(K_bath-K_o)
    return np.asarray([dVa,dDK_a_t,dK_a_s,dKg], dtype=float)

probe_cases = [
    ('CONTROL', 75, {'gki': 90.0, 'pk': 3e-4, 'd': 0.05, 'gt': 2.0, 'gs': 10.0, 'zth': 70.0, 'zs': 2.5, 'eps': 0.002, 'eps_middle': 1.0, 'wo': 1400.0, 'wo_middle': 1.0, 'ca': 500.0, 'gl_a': 5.0, 'Va_l': -70.0, 'Va_s': -92.0, 'switching_function': 'sigmoid', 'w_a': 2000.0}),
    ('MFA', 125, {'gki': 40.0, 'pk': 5e-5, 'd': 1.5, 'gt': 4.0, 'gs': 22.0, 'zth': 0.2, 'zs': 0.05, 'eps': 0.01, 'eps_middle': 0.8, 'wo': 2500.0, 'wo_middle': 1.0, 'ca': 400.0, 'gl_a': 0.01, 'Va_l': -70.0, 'Va_s': -90.0, 'switching_function': 'tanh', 'w_a': 2000.0}),
    ('MFA_BA', 100, {'gki': 25.0, 'pk': 2e-4, 'd': 1.5, 'gt': 7.0, 'gs': 14.0, 'zth': 0.2, 'zs': 0.05, 'eps': 0.01, 'eps_middle': 0.8, 'wo': 1700.0, 'wo_middle': 1.0, 'ca': 400.0, 'gl_a': 0.01, 'Va_l': -70.0, 'Va_s': -90.0, 'switching_function': 'hill', 'hill_coefficient': 3.0, 'K_d': 1.2, 'w_a': 2000.0}),
]
z = np.array([-80.0, 0.5, 0.2, 0.1], dtype=float)
probe_rows = []
for exp_type, current_na, flat in probe_cases:
    pdict = build_paramdict(exp_type, current_na, flat)
    deltas = []
    for t in [0.0, 11173.0, 12000.0, 21140.0, 22000.0]:
        got = model(z, t, pdict)
        ref = reference_model(z, t, pdict)
        deltas.append(float(np.max(np.abs(got - ref))))
    probe_rows.append({'condition': exp_type, 'current_na': current_na, 'max_abs_rhs_delta': max(deltas), 'status': 'exact_within_float_tolerance' if max(deltas) <= 1e-12 else 'mismatch'})
probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(OUTPUT_DIR / 'model_alignment_probe.csv', index=False)
display(probe_df)
assert (probe_df['status'] == 'exact_within_float_tolerance').all()

## Step 02 contract carried into Step 04

In [ ]:
region_counts = pd.read_csv(PROJECT_ROOT / 'outputs' / 'features' / 'region_condition_cell_counts.csv')
display(region_counts)
contract = acceptance_contract_table()
display(contract)

## Runtime-safe representative fit run

To keep the executed notebook auditable and stable, the notebook runs one representative six-sweep CONTROL cell directly in-kernel. The same code path was also exercised separately on representative MFA and MFA_BA cells outside the notebook; their summary is loaded below for cross-condition audit context.

In [ ]:
from src.step04_cell_fits import run_step04_cell_specific_six_sweep_fitting

results = run_step04_cell_specific_six_sweep_fitting(
    PROJECT_ROOT,
    output_dir=OUTPUT_DIR / 'notebook_control_run',
    selected_file_ids=['1_DH_1_CONTROL'],
    max_cells=1,
    n_fit_points=10,
    n_starts=1,
    max_nfev_all6=2,
    max_nfev_holdout=1,
)
summary = results['cell_fit_quality_summary']
accepted = results['accepted_cell_ensembles']
heldout = results['heldout_current_screen']
summary.to_csv(OUTPUT_DIR / 'cell_fit_quality_summary.csv', index=False)
accepted.to_csv(OUTPUT_DIR / 'accepted_cell_ensembles.csv', index=False)
heldout.to_csv(OUTPUT_DIR / 'heldout_current_screen.csv', index=False)
analysis_summary = {
    'notebook_name': '04_cell_specific_six_sweep_fitting.ipynb',
    'step_name': 'Step 04 cell-specific six-sweep fitting and accepted ensemble construction',
    'selected_file_ids': ['1_DH_1_CONTROL'],
    'n_cells': int(len(summary)),
    'n_accepted_candidates': int(len(accepted)),
    'n_reviewer_facing_cells': int(summary['cell_reviewer_facing'].sum()),
    'model_alignment': 'expected_model_exact_with_numerical_safeguards_only',
}
(OUTPUT_DIR / 'analysis_summary.json').write_text(json.dumps(analysis_summary, indent=2), encoding='utf-8')
summary

## Accepted candidates, held-out screen, and cross-condition audit context

In [ ]:
display(accepted[['file_id','condition','region','candidate_id','mean_trace_rmse_mV','mean_weighted_pass_fraction','accepted_all6']])
display(heldout[['file_id','heldout_sweep','heldout_trace_rmse_mV','heldout_weighted_pass_fraction','heldout_pass']])

extra_summaries = []
for path in [
    PROJECT_ROOT / 'outputs' / 'bench_DH_1_MFA' / 'cell_fit_quality_summary.csv',
    PROJECT_ROOT / 'outputs' / 'bench_DH_1_MFA_BA' / 'cell_fit_quality_summary.csv',
]:
    if path.exists():
        extra_summaries.append(pd.read_csv(path))
if extra_summaries:
    extra_df = pd.concat(extra_summaries, ignore_index=True)
    extra_df.to_csv(OUTPUT_DIR / 'condition_audit_summary.csv', index=False)
    display(extra_df[['file_id','condition','region','best_trace_rmse_mV','best_weighted_pass_fraction','holdout_pass_count','cell_reviewer_facing']])

## Interpretation boundary

This notebook demonstrates that Step 04 now fits the **expected reviewer-facing model** and produces cell-specific accepted ensembles under a six-sweep contract.

It does **not** by itself establish biological degeneracy.

What it supports:
- the fitted model is the expected ODE model discussed with reviewers;
- Step 04 uses one shared cell-level mechanism across six sweeps;
- held-out-sweep screening is part of the acceptance contract.

What remains for later steps:
- Step 05: mechanistic decomposition of accepted ensembles;
- Step 06: broader predictive robustness and perturbation validation;
- Step 07+: assumption sensitivity and parameter plausibility layers.